# Загрузка данных из CSV, Excel и JSON

## Версия преподавателя

Вы работаете начинающим аналитиком службы поддержки. Данные поступили из трёх систем:

- журнал обращений — `tickets.csv`;
- справочник клиентов — `clients.xlsx`;
- справочник каналов — `channels.json`.

Нужно загрузить файлы, проверить их структуру, привести ключи к совместимому виду, объединить таблицы и сохранить единый аналитический набор данных.

### Результат работы

В папке `outputs` должен появиться файл `support_tickets_datamart.csv`.

## Методическая цель

Показать начинающим слушателям, что интеграция данных состоит не только из вызова `merge`. Важно пройти четыре проверки:

1. корректно ли прочитан каждый формат;
2. совместимы ли ключи;
3. сохранилась ли гранулярность основной таблицы;
4. какие значения не нашли соответствие в справочниках.

**Контрольные значения:** 30 обращений, 10 клиентов, 5 каналов, 2 обращения без клиента, 1 обращение без канала.

## План занятия

| Этап | Время | Результат |
|---|---:|---|
| Постановка задачи | 7 минут | Понимаем, зачем нужны три файла |
| Подготовка среды | 10 минут | Проверяем Python, pandas и папки |
| Загрузка CSV | 15 минут | Получаем таблицу обращений |
| Загрузка XLSX и JSON | 13 минут | Получаем два справочника |
| Подготовка ключей | 10 минут | Убираем пробелы и согласуем типы |
| Объединение | 15 минут | Собираем единый DataFrame |
| Контроль качества | 10 минут | Проверяем строки и несовпавшие ключи |
| Сохранение | 7 минут | Записываем и повторно читаем результат |
| Итоги | 3 минуты | Проходим чек-лист |

## Сценарий проведения

- После каждой загрузки останавливайтесь и просите группу назвать размер таблицы.
- Перед `merge` спросите: «Какая таблица является основной и почему?»
- Не исправляйте неизвестные ключи автоматически: они нужны для демонстрации контроля качества.
- После `merge` сначала сравните количество строк, затем ищите пропуски.
- Мини-задание можно дать на 8–10 минут самостоятельной работы.

## 1. Подготовка среды

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display

print("Версия Python:", sys.version.split()[0])
print("Версия pandas:", pd.__version__)
print("Текущая рабочая папка:", Path.cwd())

Версия Python: 3.10.5
Версия pandas: 2.0.1
Текущая рабочая папка: d:\Андрей\РАБОТА ФИН.УНИВЕР. ПРЕПОД\Занятие 3 18.07.2026\Пара 1\loading_and_integration


In [2]:
# Notebook расположен в корне учебного проекта.
# Все пути относительные, поэтому проект работает и в Google Colab, и в VS Code.
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TICKETS_PATH = RAW_DIR / "tickets.csv"
CLIENTS_PATH = RAW_DIR / "clients.xlsx"
CHANNELS_PATH = RAW_DIR / "channels.json"
OUTPUT_PATH = OUTPUT_DIR / "support_tickets_datamart.csv"

print("Папка исходных данных:", RAW_DIR)
print("Папка результатов:", OUTPUT_DIR)

Папка исходных данных: d:\Андрей\РАБОТА ФИН.УНИВЕР. ПРЕПОД\Занятие 3 18.07.2026\Пара 1\loading_and_integration\data\raw
Папка результатов: d:\Андрей\РАБОТА ФИН.УНИВЕР. ПРЕПОД\Занятие 3 18.07.2026\Пара 1\loading_and_integration\outputs


In [3]:
# Резервный сценарий для Google Colab.
# Если исходные файлы не загружены вместе с notebook, эта ячейка создаст их автоматически.

tickets_rows = [
    ["T001","2026-05-01 09:10","C001","CH01","Доступ к аккаунту","Высокий","Resolved",2.5],
    ["T002","2026-05-01 10:25","C002","CH02","Оплата","Средний","Resolved",5.0],
    ["T003","2026-05-01 11:40","C003","CH03","Техническая ошибка","Высокий","In Progress",None],
    ["T004","2026-05-02 08:15"," C004 ","CH01","Доставка","Низкий","Resolved",8.0],
    ["T005","2026-05-02 13:30","C005","CH04","Возврат","Средний","Open",None],
    ["T006","2026-05-03 09:05","C006"," CH02 ","Оплата","Высокий","Resolved",3.5],
    ["T007","2026-05-03 12:20","C007","CH03","Консультация","Низкий","Resolved",1.0],
    ["T008","2026-05-04 14:00","C008","CH05","Техническая ошибка","Высокий","Resolved",12.0],
    ["T009","2026-05-04 15:45","C009","CH01","Доступ к аккаунту","Средний","Resolved",4.0],
    ["T010","2026-05-05 09:35","C010","CH02","Доставка","Низкий","Open",None],
    ["T011","2026-05-05 11:10","C001","CH03","Оплата","Средний","Resolved",6.5],
    ["T012","2026-05-06 10:00","C002","CH04","Возврат","Высокий","Resolved",15.0],
    ["T013","2026-05-06 16:25","C003","CH05","Консультация","Низкий","Resolved",0.8],
    ["T014","2026-05-07 08:50","C004","CH01","Техническая ошибка","Высокий","In Progress",None],
    ["T015","2026-05-07 13:15","C005","CH02","Доступ к аккаунту","Средний","Resolved",2.0],
    ["T016","2026-05-08 09:45","C006","CH03","Доставка","Низкий","Resolved",7.0],
    ["T017","2026-05-08 12:35","C007","CH04","Оплата","Средний","Resolved",4.5],
    ["T018","2026-05-09 10:20","C008","CH05","Возврат","Высокий","Open",None],
    ["T019","2026-05-09 14:30","C009","CH01","Консультация","Низкий","Resolved",1.5],
    ["T020","2026-05-10 09:00","C010","CH02","Техническая ошибка","Высокий","Resolved",10.0],
    ["T021","2026-05-10 11:55","C001","CH03","Доставка","Средний","Resolved",6.0],
    ["T022","2026-05-11 08:40","C002","CH04","Доступ к аккаунту","Высокий","Resolved",3.0],
    ["T023","2026-05-11 13:20","C003","CH05","Оплата","Средний","In Progress",None],
    ["T024","2026-05-12 10:15","C004","CH01","Возврат","Низкий","Resolved",9.0],
    ["T025","2026-05-12 15:05","C005","CH02","Консультация","Низкий","Resolved",1.2],
    ["T026","2026-05-13 09:25","C006","CH03","Техническая ошибка","Высокий","Resolved",11.0],
    ["T027","2026-05-13 12:10","C007","CH04","Доставка","Средний","Resolved",5.5],
    ["T028","2026-05-14 08:30","C999","CH05","Доступ к аккаунту","Высокий","Resolved",2.8],
    ["T029","2026-05-14 14:45","C008","CH99","Оплата","Средний","Resolved",4.2],
    ["T030","2026-05-15 10:05","C999","CH01","Возврат","Высокий","Open",None],
]

clients_rows = [
    ["C001","B2C","Москва","2025-01-15"],
    ["C002","B2B","Санкт-Петербург","2025-02-20"],
    ["C003","B2C","Казань","2025-03-05"],
    ["C004","B2B","Екатеринбург","2025-03-18"],
    ["C005","B2C","Москва","2025-04-01"],
    ["C006","B2G","Новосибирск","2025-04-14"],
    ["C007","B2C","Самара","2025-05-02"],
    ["C008","B2B","Москва","2025-05-20"],
    ["C009","B2C","Ростов-на-Дону","2025-06-10"],
    ["C010","B2G","Нижний Новгород","2025-06-28"],
]

channels_rows = [
    ["CH01","Телефон",False],
    ["CH02","Электронная почта",True],
    ["CH03","Чат",True],
    ["CH04","Мобильное приложение",True],
    ["CH05","Офис",False],
]

if not TICKETS_PATH.exists():
    pd.DataFrame(
        tickets_rows,
        columns=["ticket_id","created_at","client_id","channel_code","category","priority","status","resolution_hours"]
    ).to_csv(TICKETS_PATH, index=False, encoding="utf-8-sig")
    print("Создан:", TICKETS_PATH)

if not CLIENTS_PATH.exists():
    pd.DataFrame(
        clients_rows,
        columns=["client_id","client_segment","city","registration_date"]
    ).to_excel(CLIENTS_PATH, index=False, sheet_name="clients")
    print("Создан:", CLIENTS_PATH)

if not CHANNELS_PATH.exists():
    pd.DataFrame(
        channels_rows,
        columns=["channel_code","channel_name","is_digital"]
    ).to_json(CHANNELS_PATH, orient="records", force_ascii=False, indent=2)
    print("Создан:", CHANNELS_PATH)

In [4]:
required_files = [TICKETS_PATH, CLIENTS_PATH, CHANNELS_PATH]

print("Проверяем исходные файлы:")
for file_path in required_files:
    status = "OK" if file_path.exists() else "НЕ НАЙДЕН"
    print(f"{status}: {file_path}")

assert all(path.exists() for path in required_files), "Не все исходные файлы доступны."

Проверяем исходные файлы:
OK: d:\Андрей\РАБОТА ФИН.УНИВЕР. ПРЕПОД\Занятие 3 18.07.2026\Пара 1\loading_and_integration\data\raw\tickets.csv
OK: d:\Андрей\РАБОТА ФИН.УНИВЕР. ПРЕПОД\Занятие 3 18.07.2026\Пара 1\loading_and_integration\data\raw\clients.xlsx
OK: d:\Андрей\РАБОТА ФИН.УНИВЕР. ПРЕПОД\Занятие 3 18.07.2026\Пара 1\loading_and_integration\data\raw\channels.json


## 2. Загрузка и проверка CSV

In [5]:
# dtype помогает сразу сохранить идентификаторы как текст.
tickets = pd.read_csv(
    TICKETS_PATH,
    dtype={"ticket_id": "string", "client_id": "string", "channel_code": "string"}
)

display(tickets.head())
print("Размер tickets:", tickets.shape)

,ticket_id,created_at,client_id,channel_code,category,priority,status,resolution_hours
0,T001,2026-05-01 09:10,C001,CH01,Доступ к аккаунту,Высокий,Resolved,2.5
1,T002,2026-05-01 10:25,C002,CH02,Оплата,Средний,Resolved,5.0
2,T003,2026-05-01 11:40,C003,CH03,Техническая ошибка,Высокий,In Progress,NaN
3,T004,2026-05-02 08:15,C004,CH01,Доставка,Низкий,Resolved,8.0
4,T005,2026-05-02 13:30,C005,CH04,Возврат,Средний,Open,NaN


Размер tickets: (30, 8)


In [ ]:
print("Столбцы:")
print(tickets.columns.tolist())

print("\nТипы данных:")
display(tickets.dtypes.to_frame("dtype"))

print("\nПропуски:")
display(tickets.isna().sum().to_frame("missing_values"))

print("\nДубликаты ticket_id:", tickets["ticket_id"].duplicated().sum())

In [ ]:
# errors="coerce" превращает нераспознанные значения в NaT.
tickets["created_at"] = pd.to_datetime(tickets["created_at"], errors="coerce")

print("Тип created_at:", tickets["created_at"].dtype)
print("Нераспознанных дат:", tickets["created_at"].isna().sum())

### Что проговорить

`head()` отвечает только на вопрос «как выглядят несколько первых строк». Для контроля качества нужны также `shape`, `dtypes`, `isna()` и `duplicated()`.

## 3. Загрузка Excel

In [ ]:
clients = pd.read_excel(
    CLIENTS_PATH,
    sheet_name="clients",
    dtype={"client_id": "string"}
)
clients["registration_date"] = pd.to_datetime(clients["registration_date"], errors="coerce")

display(clients.head())
print("Размер clients:", clients.shape)
print("Дубликаты client_id:", clients["client_id"].duplicated().sum())

### Контрольная точка

- 10 строк;
- 4 столбца;
- `client_id` уникален;
- `registration_date` имеет тип `datetime64[ns]`.

## 4. Загрузка JSON

In [ ]:
channels = pd.read_json(CHANNELS_PATH, dtype={"channel_code": "string"})

display(channels)
print("Размер channels:", channels.shape)
print("Дубликаты channel_code:", channels["channel_code"].duplicated().sum())

### Контрольная точка

- 5 строк;
- 3 столбца;
- `channel_code` уникален;
- поле `is_digital` логическое.

## 5. Подготовка и сравнение ключей

In [ ]:
# Удаляем пробелы по краям. Это важно: " C004 " и "C004" — разные строки.
for column in ["client_id", "channel_code"]:
    tickets[column] = tickets[column].astype("string").str.strip()

clients["client_id"] = clients["client_id"].astype("string").str.strip()
channels["channel_code"] = channels["channel_code"].astype("string").str.strip()

print("Пример client_id после очистки:", tickets.loc[tickets["ticket_id"] == "T004", "client_id"].iloc[0])
print("Пример channel_code после очистки:", tickets.loc[tickets["ticket_id"] == "T006", "channel_code"].iloc[0])

In [ ]:
unknown_clients_before_merge = sorted(
    set(tickets["client_id"].dropna()) - set(clients["client_id"].dropna())
)
unknown_channels_before_merge = sorted(
    set(tickets["channel_code"].dropna()) - set(channels["channel_code"].dropna())
)

print("Клиенты, которых нет в справочнике:", unknown_clients_before_merge)
print("Каналы, которых нет в справочнике:", unknown_channels_before_merge)

### Ожидаемый ответ группы

В данных есть клиент `C999`, которого нет в справочнике, и канал `CH99`, которого нет в справочнике. Это не техническая ошибка pandas, а проблема соответствия данных.

## 6. Первое объединение

In [ ]:
# left означает: сохраняем все обращения из основной таблицы tickets.
tickets_with_clients = tickets.merge(
    clients,
    on="client_id",
    how="left",
    validate="many_to_one"
)

print("Строк до объединения:", len(tickets))
print("Строк после добавления клиентов:", len(tickets_with_clients))
display(tickets_with_clients.head())

### Почему используется `validate="many_to_one"`

Много обращений могут относиться к одному клиенту, но один идентификатор клиента должен встречаться в справочнике только один раз. Если это правило нарушено, pandas остановит объединение с понятной ошибкой.

## 7. Второе объединение

In [ ]:
tickets_datamart = tickets_with_clients.merge(
    channels,
    on="channel_code",
    how="left",
    validate="many_to_one"
)

print("Строк после добавления каналов:", len(tickets_datamart))
print("Столбцов в итоговой таблице:", tickets_datamart.shape[1])
display(tickets_datamart.head())

## 8. Контроль качества

In [ ]:
quality_report = pd.DataFrame({
    "check": [
        "Строк в исходной таблице",
        "Строк в итоговой таблице",
        "Уникальных ticket_id",
        "Обращений без найденного клиента",
        "Обращений без найденного канала",
        "Нераспознанных дат"
    ],
    "value": [
        len(tickets),
        len(tickets_datamart),
        tickets_datamart["ticket_id"].nunique(),
        tickets_datamart["client_segment"].isna().sum(),
        tickets_datamart["channel_name"].isna().sum(),
        tickets_datamart["created_at"].isna().sum()
    ]
})

display(quality_report)

assert len(tickets_datamart) == len(tickets), "После merge изменилось количество строк."
assert tickets_datamart["ticket_id"].nunique() == len(tickets), "ticket_id перестал быть уникальным."

In [ ]:
print("Обращения с неизвестным клиентом:")
display(
    tickets_datamart.loc[
        tickets_datamart["client_segment"].isna(),
        ["ticket_id", "client_id", "category", "status"]
    ]
)

print("Обращения с неизвестным каналом:")
display(
    tickets_datamart.loc[
        tickets_datamart["channel_name"].isna(),
        ["ticket_id", "channel_code", "category", "status"]
    ]
)

### Главный методический акцент

`merge` завершился без исключения, но три строки получили пропуски из справочников. Поэтому «код работает» и «данные интегрированы корректно» — не одно и то же.

## 9. Простые аналитические расчёты

In [ ]:
print("Количество обращений по каналам:")
display(
    tickets_datamart["channel_name"]
    .fillna("Неизвестный канал")
    .value_counts()
    .rename_axis("channel_name")
    .reset_index(name="tickets_count")
)

print("Среднее время решения по сегментам:")
display(
    tickets_datamart
    .groupby("client_segment", dropna=False, as_index=False)
    .agg(
        tickets_count=("ticket_id", "nunique"),
        avg_resolution_hours=("resolution_hours", "mean")
    )
    .sort_values("avg_resolution_hours", ascending=False)
)

## 10. Сохранение и повторное чтение

In [ ]:
tickets_datamart.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("Итоговый файл сохранён:", OUTPUT_PATH)
print("Размер файла, байт:", OUTPUT_PATH.stat().st_size)

In [ ]:
result_check = pd.read_csv(OUTPUT_PATH)
print("Размер повторно загруженного файла:", result_check.shape)
display(result_check.head())

assert result_check.shape == tickets_datamart.shape, "Размер сохранённого файла отличается от DataFrame."

## 11. Решение мини-задания

In [ ]:
resolved_by_segment = (
    tickets_datamart
    .loc[tickets_datamart["status"] == "Resolved"]
    .groupby("client_segment", dropna=False, as_index=False)
    .agg(
        resolved_tickets=("ticket_id", "nunique"),
        avg_resolution_hours=("resolution_hours", "mean")
    )
    .sort_values("avg_resolution_hours", ascending=False)
)

display(resolved_by_segment)

mini_output_path = OUTPUT_DIR / "resolved_tickets_by_segment.csv"
resolved_by_segment.to_csv(mini_output_path, index=False, encoding="utf-8-sig")
print("Мини-задание сохранено:", mini_output_path)

### Ожидаемый аналитический вывод

Среди известных сегментов максимальное среднее время решения у B2G. Следует помнить, что расчёт основан только на решённых обращениях и на небольшом учебном наборе данных.

## 12. Автоматическая проверка контрольных значений

In [ ]:
# Контрольные значения для быстрой проверки преподавателем.
expected = {
    "tickets_rows": 30,
    "clients_rows": 10,
    "channels_rows": 5,
    "datamart_rows": 30,
    "datamart_columns": 13,
    "unknown_client_rows": 2,
    "unknown_channel_rows": 1,
    "resolved_tickets": 23,
}

actual = {
    "tickets_rows": len(tickets),
    "clients_rows": len(clients),
    "channels_rows": len(channels),
    "datamart_rows": len(tickets_datamart),
    "datamart_columns": tickets_datamart.shape[1],
    "unknown_client_rows": int(tickets_datamart["client_segment"].isna().sum()),
    "unknown_channel_rows": int(tickets_datamart["channel_name"].isna().sum()),
    "resolved_tickets": int((tickets_datamart["status"] == "Resolved").sum()),
}

check_table = pd.DataFrame({"expected": expected, "actual": actual})
check_table["ok"] = check_table["expected"] == check_table["actual"]
display(check_table)

assert check_table["ok"].all(), "Одно или несколько контрольных значений не совпали."

## 13. Типовые затруднения группы

| Ситуация | Диагностика | Подсказка преподавателю |
|---|---|---|
| Не найден файл | Показать `Path.cwd()` и содержимое `data/raw` | Попросить повторно выполнить блок создания данных |
| Не работает Excel | Проверить `openpyxl` | В Colab установить библиотеку, локально — из `requirements.txt` |
| Слушатель использует `inner` | Число строк уменьшается | Спросить, должны ли исчезать обращения без справочника |
| После merge больше строк | Нарушена уникальность справочника | Показать `duplicated()` и `validate` |
| Пропуски воспринимаются как ошибка кода | Показать неизвестные ключи | Разделить техническую и содержательную диагностику |

## 14. Задание повышенной сложности

Для быстрых слушателей:

1. создать поле `resolution_days = resolution_hours / 24`;
2. добавить признак `is_resolved`;
3. посчитать долю решённых обращений по каналам;
4. отдельно показать канал `Неизвестный канал`;
5. сохранить результат в `outputs/channel_resolution_summary.csv`.

## 15. Завершение занятия

Попросите двух слушателей кратко объяснить:

1. почему основной таблицей были обращения;
2. что проверяется после `merge`;
3. чем неизвестный ключ отличается от ошибки выполнения кода.

После этого группа фиксирует итоговый маршрут: загрузка → проверка → ключи → объединение → контроль → экспорт.